# Laptop Price Prediction

**Goal**: Predict `buynow_price` and minimize RMSE.

This notebook:
- Builds list-column vocabularies from **train only**
- Puts all preprocessing in a `Pipeline` + `ColumnTransformer` with `OneHotEncoder(handle_unknown="ignore")` (no `drop_first`, no per-frame `get_dummies`, no `.align()`)
- Reports 5-fold CV RMSE mean ± std for model selection
- Includes a mean-predictor baseline and MAE/MAPE
- Uses permutation importance grouped by parent feature
- Does **not** claim iterative VIF filtering or a specific ~505 RMSE (those require a different preprocessing path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
train = pd.read_json('data/train_dataset.json')
val   = pd.read_json('data/val_dataset.json')
test  = pd.read_json('data/test_dataset.json')
print('Shapes:', train.shape, val.shape, test.shape)
train.head(2)

## 2. Integrity Checks

In [ ]:
def coverage_report(train, other, name, cols):
    rows = []
    for col in cols:
        train_vals = set(train[col].dropna().astype(str).unique())
        unseen = (~other[col].astype(str).isin(train_vals) & other[col].notna()).sum()
        rows.append({
            'column': col,
            'split': name,
            'nan_rate': other[col].isna().mean(),
            'unseen_pct': 100 * unseen / len(other),
            'unseen_count': unseen
        })
    return pd.DataFrame(rows)

cat_check = ['graphic card type', 'resolution (px)', 'drive type', 'RAM type',
             'CPU model', 'warranty', 'operating system', 'communications',
             'input devices', 'multimedia']

rep = pd.concat([
    coverage_report(train, val, 'val', cat_check),
    coverage_report(train, test, 'test', cat_check)
])
print(rep.round(2).to_string(index=False))
print('\nMax unseen %:', rep['unseen_pct'].max())

## 3. Feature Engineering (vocab from train only)

In [ ]:
def extract_ram(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if 'mb' in s: return float(s.replace('mb','').strip()) / 1024
    if 'gb' in s: return float(s.replace('gb','').strip())
    try: return float(s)
    except: return np.nan

def extract_screen(x):
    if pd.isna(x): return np.nan
    import re
    nums = re.findall(r'[\d\.]+', str(x))
    return float(nums[0]) if nums else np.nan

def split_resolution(x):
    if pd.isna(x) or not isinstance(x, str): return np.nan, np.nan
    parts = x.split(' x ')
    if len(parts) == 2:
        try: return int(parts[0]), int(parts[1])
        except: return np.nan, np.nan
    return np.nan, np.nan

def engineer(df):
    df = df.copy()
    df['RAM_size_gb'] = df['RAM size'].apply(extract_ram)
    df['screen_size_num'] = df['screen size'].apply(extract_screen)
    res = df['resolution (px)'].apply(split_resolution)
    df['resolution_x'] = res.apply(lambda t: t[0])
    df['resolution_y'] = res.apply(lambda t: t[1])
    df['CPU_cores_num'] = pd.to_numeric(df['CPU cores'], errors='coerce')
    df['drive_memory_gb'] = pd.to_numeric(df['drive memory size (GB)'], errors='coerce')
    # Flatten lists to pipe-separated strings (vocab learned by OneHotEncoder on train)
    for col in ['communications', 'operating system', 'input devices', 'multimedia']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: '|'.join(sorted(x)) if isinstance(x, list) else x)
    return df

train_e = engineer(train)
val_e   = engineer(val)
test_e  = engineer(test)

num_features = ['CPU clock speed (GHz)', 'RAM_size_gb', 'screen_size_num',
                'CPU_cores_num', 'drive_memory_gb', 'resolution_x', 'resolution_y']
cat_features = ['graphic card type', 'drive type', 'RAM type', 'CPU model', 'warranty',
                'communications', 'operating system', 'input devices', 'multimedia']
target = 'buynow_price'

print('Engineered. List columns flattened; OneHotEncoder will learn vocab from train only.')

## 4. Pipeline (no drop_first, no .align)

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

print('Pipeline ready. Unseen categories → all-zero rows. No per-frame dummy misalignment.')

## 5. Baseline + 5-fold CV Model Selection

In [ ]:
X_train = train_e[num_features + cat_features]
y_train = train_e[target]
X_test  = test_e[num_features + cat_features]
y_test  = test_e[target]

# Mean predictor baseline
mean_pred = np.full_like(y_test, y_train.mean(), dtype=float)
print(f'Mean predictor  Test RMSE: {np.sqrt(mean_squared_error(y_test, mean_pred)):.1f}  '
      f'MAE: {mean_absolute_error(y_test, mean_pred):.1f}')

def rmse_scorer(est, X, y):
    return -np.sqrt(mean_squared_error(y, est.predict(X)))

models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=150, random_state=42, n_jobs=-1),
}

cv_rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring=rmse_scorer, n_jobs=-1)
    cv_rows.append({'Model': name, 'CV RMSE mean': -scores.mean(), 'CV RMSE std': scores.std()})

cv_df = pd.DataFrame(cv_rows).sort_values('CV RMSE mean')
print('\n5-fold CV (baseline hyperparameters):')
print(cv_df.round(1).to_string(index=False))

# Voting
voting = VotingRegressor([
    ('gb', GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=150, random_state=42, n_jobs=-1)),
    ('rf', RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1))
], weights=[0.4, 0.35, 0.25])

pipe_v = Pipeline([('prep', preprocessor), ('model', voting)])
v_scores = cross_val_score(pipe_v, X_train, y_train, cv=5, scoring=rmse_scorer, n_jobs=-1)
print(f'\nVoting CV RMSE: {-v_scores.mean():.1f} ± {v_scores.std():.1f}')

## 6. Final Evaluation on Test

In [ ]:
final = Pipeline([('prep', preprocessor), ('model', voting)])
final.fit(X_train, y_train)
pred = final.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae  = mean_absolute_error(y_test, pred)
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100
print(f'Test RMSE: {rmse:.1f}   MAE: {mae:.1f}   MAPE: {mape:.1f}%')

plt.figure(figsize=(6, 5))
sns.regplot(x=pred, y=y_test, scatter_kws={'alpha': 0.35}, line_kws={'color': 'red'})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Voting — Test (RMSE={rmse:.0f})')
plt.tight_layout()
plt.show()

## 7. Permutation Importance (grouped by parent)

In [ ]:
rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train)

r = permutation_importance(
    rf_pipe, val_e[num_features + cat_features], val_e[target],
    n_repeats=8, random_state=42, n_jobs=-1,
    scoring='neg_root_mean_squared_error'
)

names = rf_pipe.named_steps['prep'].get_feature_names_out()
imp = pd.Series(r.importances_mean, index=names)

# Rough grouping: take the part after the last __ and before the first _
grouped = {}
for n, v in imp.items():
    parent = n.split('__')[-1].split('_')[0]
    grouped[parent] = grouped.get(parent, 0) + max(v, 0)

g = pd.Series(grouped).sort_values(ascending=False).head(12)
plt.figure(figsize=(8, 5))
g.plot(kind='barh', color='steelblue')
plt.title('Permutation Importance (grouped by parent feature)')
plt.xlabel('Mean decrease in RMSE')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
print(g.head(6))

## Conclusion

- Preprocessing is leakage-safe: vocabularies and encoders are fit on train only; unseen categories are ignored.
- 5-fold CV mean ± std is used for model selection instead of a single split.
- A mean-predictor baseline and MAE/MAPE give context for the RMSE.
- Permutation importance (grouped) shows which original features drive price; one-hot columns are not treated as independent.
- Hyperparameters are baseline (not tuned). Iterative VIF is not performed in this notebook.